In [ ]:
from src.requirements import *
from src.audio_handler import AudioDataset, collate_padding
from src.ssl_model import *
from src.ssl_large import *
from src.dataset import *
from src.metrics import *
import logging
from collections import defaultdict


In [ ]:
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = True   # RTX 4090: safe + fast
torch.backends.cudnn.allow_tf32 = True
use_bfloat = True

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

In [ ]:
train_path = os.path.join('data', 'metadata_normal.tsv')
val_path = os.path.join('data', 'metadata_val_normal.tsv')
cache = os.path.join('data', 'cache')
save_dir = os.path.join('checkpoints', 'ssl')

In [ ]:
def get_lr(step: int, warmup_steps: int, total_steps: int, base_lr: float,
           min_lr: float = 1e-6) -> float:
    if step < warmup_steps:
        return base_lr * step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    cosine = 0.5 * (1 + math.cos(math.pi * progress))
    return min_lr + (base_lr - min_lr) * cosine

In [ ]:
def save_checkpoint(model, optimizer, scaler, step, loss, cfg, save_dir):
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = save_dir / f"checkpoint_{step:07d}.pt"
    torch.save({
        "step":          step,
        "model":         model.state_dict(),
        "optimizer":     optimizer.state_dict(),
        "scaler":        scaler.state_dict(),
        "loss":          loss,
        "config":        cfg.__dict__,
    }, ckpt_path)
    # Also save 'latest' symlink
    latest = save_dir / "latest.pt"
    if latest.is_symlink() or latest.exists():
        latest.unlink()
    latest.symlink_to(ckpt_path.name)
    log.info(f"Saved checkpoint → {ckpt_path}")
    return ckpt_path

def load_checkpoint(model, optimizer, scaler, path, device):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scaler.load_state_dict(ckpt["scaler"])
    log.info(f"Resumed from step {ckpt['step']}, loss={ckpt['loss']:.4f}")
    return ckpt["step"]

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
log.info(f"Device: {device}")

lr = 5e-4
weight_decay = 0.01
batch_size = 16
grad_accum = 4
# effective batch_size = 16 * 4 = 64 (<- this is good for contrastive loss)
total_updates = 100_000

config = NepaliSSLConfig(
    hidden_dim=512,
    num_heads=6,
    num_layers=6,
    ffn_dim=2048,
    dropout=0.1,
    logit_temp=0.1,
    mask_prob=0.15,
    mask_length=10,
    num_negatives=100,
    diversity_loss_weight=0.1,
    commitment_loss_weight=0.25,
    codebook_temp_start=2.0,
    codebook_dim=0.5,
    codebook_temp_anneal_steps=100_000
)

model = NepaliSSL(config).to(device)
count_parameters(model)

if hasattr(torch, "compile"):
    log.info("Compiling model with torch.compile")
    model = torch.compile(model, mode="reduce-overhead")
    
model.context.encoder.gradient_checkpointing_enable() \
    if hasattr(model.context.encoder, "gradient_checkpointing_enable") else None
    
decay, no_decay = [], []

for name, param in model.named_parameters():
    if param.requires_grad:
        if "norm" in name or "bias" in name or "pe" in name or "mask_emb" in name:
            no_decay.append(param)
        else:
            decay.append(param)

optimizer = torch.optim.AdamW([
            {"params": decay,    "weight_decay": weight_decay},
            {"params": no_decay, "weight_decay": 0.0},
        ], lr=lr, betas=(0.9, 0.98), eps=1e-6)

scaler = torch.GradScaler(device=device, enabled=use_bfloat)

train_dataset = NepaliSpeechDataset(metadata_path=train_path, audio_base_dir='data/audio', use_memory_map=True, cache_dir=cache, for_ssl=True)
val_dataset = NepaliSpeechDataset(metadata_path=val_path, audio_base_dir='data/audio', use_memory_map=True, cache_dir=cache, for_ssl=True)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    pin_memory=True,
    collate_fn=collate_fn_ssl,
    persistent_workers=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    pin_memory=True,
    collate_fn=collate_fn_ssl,
    persistent_workers=True,
)

step = 0
metrics_history = defaultdict(list)
last_features = None # cached for dead code reset

resume = False
if resume:
    step = load_checkpoint(model, optimizer, scaler, resume, device)

In [ ]:
def evaluate(model, eval_loader, device, step, metrics_history, n_samples=1000):
    log.info("Running evaluation")
    model.eval()
    results = compute_feature_metrics(model, eval_loader, device, n_samples=n_samples)
    model.train()
    log.info(f"[Eval step={step}] {results}")
    for k, v in results.items():
        metrics_history[f"eval_{k}"].append(v)

In [ ]:
def train(total_updates, warmup_steps, grad_accum, model, optimizer, val_loader, train_loader, scaler, device, step, metrics_history, config):
    total_steps = total_updates
    warmup_steps = warmup_steps
    accum_steps = grad_accum
    max_grad_norm = 1.0
    save_every = 5_000
    log_interval = 100
    eval_interval = 1_000

    model.train()
    optimizer.zero_grad()

    loader_iter = iter(train_loader)
    accum_loss = 0.0
    t0 = time.time()
    
    tau = max(
        config.codebook_temp_end,
        config.codebook_temp_start - 
        (config.codebook_temp_start - config.codebook_temp_end) 
        * (step / config.codebook_temp_anneal_steps)
    )

    while step < total_steps:
        
        # Gradient accumulation
        for _ in range(accum_steps):
            try:
                batch = next(loader_iter)
            except StopIteration:
                loader_iter = iter(train_loader)
                batch = next(loader_iter)

            waveform = batch["waveform"].to(device, non_blocking=True)
            lengths  = batch["lengths"].to(device, non_blocking=True)

            dtype = torch.bfloat16 if use_bfloat else torch.float32
            with torch.autocast(device_type="cuda", dtype=dtype, enabled=use_bfloat):
                out = model(waveform, lengths, step=step)
                loss = out["loss"] / accum_steps

            scaler.scale(loss).backward()
            
            accum_loss += loss.item()
            accum_contrast += out["loss_contrast"] / accum_steps
            accum_div     += out["loss_diversity"] / accum_steps
            accum_commit  += out["loss_commitment"] / accum_steps
            
            last_features = out["_features"]

        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)

        # LR update
        lr = get_lr(step, warmup_steps, total_steps, lr)
        for g in optimizer.param_groups:
            g["lr"] = lr

        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        step += 1

        # Dead code reset every 1000 steps
        if step % 1000 == 0 and last_features is not None:
            flat_feats = last_features.reshape(-1, last_features.size(-1))
            model.quantizer.reset_dead_codes(flat_feats, threshold=2)

        # Logging
        if step % log_interval == 0:
            elapsed = time.time() - t0
            ups = log_interval / elapsed
            tau = model.get_tau(step)
            log.info(
                f"step={step:>7d} | "
                f"loss={accum_loss:.4f} | "
                f"contrast={accum_contrast:.4f} | "
                f"diversity={accum_div:.4f} | "
                f"commit={accum_commit:.4f} | "
                f"perp={out['perplexity']:.1f} | "
                f"tau={tau:.3f} | "
                f"lr={lr:.2e} | "
                f"{ups:.1f} ups/s"
            )
            metrics_history["step"].append(step)
            metrics_history["loss"].append(accum_loss)
            metrics_history["perplexity"].append(out["perplexity"])
            metrics_history["tau"].append(tau)
            accum_loss = accum_contrast = accum_div = accum_commit = 0.0
            t0 = time.time()

        # Checkpoint
        if step % save_every == 0:
            save_checkpoint(
                model, optimizer, scaler,
                step, out["loss"].item(), config, save_dir
            )

        # Eval metrics
        if step % eval_interval == 0:
            evaluate(model, val_loader, device, step, metrics_history)

    log.info("Training complete.")
    save_checkpoint(
        model, optimizer, scaler,
        step, out["loss"].item(), config, save_dir
    )
    
    # Save metrics history
    with open(Path(save_dir) / "metrics.json", "w") as f:
        json.dump(metrics_history, f, indent=2)